# Session 3: when a model learns the noise
**A. Real-world problem.** A model with enough knobs can pass through every training point and still
predict next season badly. Today we measure that (train versus test error), tame it (ridge
regression), choose settings honestly (k-fold cross-validation), see why (bias and variance),
defend against bad data points (robust lines), and meet two traps: correlation that is not
causation, and two inputs that say the same thing. All data are synthetic, so we know the truth.
This is general education about modelling, not agronomic advice.

**Plan.** (1) polynomial degree and the U-shaped test error, (2) ridge, (3) cross-validation,
(4) bias-variance, (5) robust lines, (6) confounding, (7) multicollinearity, (8) coefficient
recovery, (9) mini-project: audit a farm yield model.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

SIGMA = 0.30


def truth(z):
    """True yield response (t/ha) to scaled rainfall z in [-1, 1]."""
    return 3.0 + 1.5 * np.sin(2.5 * np.asarray(z, float))


def make_response(n, seed, sigma=SIGMA):
    rng = np.random.default_rng(seed)
    z = rng.uniform(-1.0, 1.0, n)
    return z, truth(z) + rng.normal(0.0, sigma, n)

## Step 1: polynomial degree and overfitting
A degree-d polynomial has d + 1 coefficients. Low degree cannot bend enough (high *bias*); high
degree bends to follow the noise (high *variance*). We fit on 20 training seasons and score on 500
fresh ones. The error measure is the root mean squared error (RMSE, t/ha). Training error only
ever falls as d grows; test error falls and then rises: a U shape.

In [ ]:
def poly_design(z, degree):
    """Columns 1, z, z^2, ..., z^degree."""
    return np.vander(np.asarray(z, float), degree + 1, increasing=True)


def fit_poly(z, y, degree):
    return np.linalg.lstsq(poly_design(z, degree), np.asarray(y, float), rcond=None)[0]


def rmse(y, yhat):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(yhat, float)) ** 2)))


def degree_curve(max_degree=12, n_train=20, seed=1):
    """Train and test RMSE for degrees 0..max_degree."""
    zt, yt = make_response(n_train, seed)
    zs, ys = make_response(500, seed + 1000)
    tr, te = [], []
    for d in range(max_degree + 1):
        c = fit_poly(zt, yt, d)
        tr.append(rmse(yt, poly_design(zt, d) @ c))
        te.append(rmse(ys, poly_design(zs, d) @ c))
    return np.array(tr), np.array(te)

## Step 2: ridge regression and standardising
Ridge adds a penalty lambda * (sum of squared coefficients) to the squared miss, shrinking the
coefficients. The intercept is not penalised, and because the penalty treats all coefficients
alike we first **standardise** each feature (subtract its mean, divide by its standard
deviation) so no feature wins just because of its units. Closed form, on centred data:

    beta = (X^T X + lambda I)^-1 X^T y.

With lambda = 0 this is ordinary least squares.

In [ ]:
def standardise(F, mean=None, sd=None):
    F = np.asarray(F, float)
    mean = F.mean(axis=0) if mean is None else mean
    sd = F.std(axis=0) if sd is None else sd
    return (F - mean) / sd, mean, sd


def ridge_fit(F, y, lam):
    """Ridge on raw features F (no constant column). Returns a dict that ridge_predict can use."""
    Fs, mean, sd = standardise(F)
    ym = float(np.mean(y))
    p = Fs.shape[1]
    beta = np.linalg.solve(Fs.T @ Fs + lam * np.eye(p), Fs.T @ (np.asarray(y, float) - ym))
    return {"beta": beta, "mean": mean, "sd": sd, "ym": ym}


def ridge_predict(model, F):
    Fs = (np.asarray(F, float) - model["mean"]) / model["sd"]
    return model["ym"] + Fs @ model["beta"]


def poly_features(z, degree):
    """z, z^2, ..., z^degree (no constant column)."""
    return poly_design(z, degree)[:, 1:]


def ridge_path(lams, degree=9, n_train=20, seed=1):
    """Largest |standardised coefficient| and test RMSE for each lambda."""
    zt, yt = make_response(n_train, seed)
    zs, ys = make_response(500, seed + 1000)
    big, te = [], []
    for lam in lams:
        m = ridge_fit(poly_features(zt, degree), yt, lam)
        big.append(float(np.abs(m["beta"]).max()))
        te.append(rmse(ys, ridge_predict(m, poly_features(zs, degree))))
    return np.array(big), np.array(te)


def expected_test_rmse(predict, sigma=SIGMA):
    """Population test RMSE (t/ha): sqrt(noise variance + mean squared gap between model and truth over z in [-1, 1])."""
    g = np.linspace(-1.0, 1.0, 2001)
    return float(np.sqrt(sigma ** 2 + np.mean((predict(g) - truth(g)) ** 2)))

## Step 3: k-fold cross-validation
We cannot use the test data to choose settings (that would leak). Instead split the training
seasons into k folds; train on k - 1 of them, score on the held-out fold, rotate, and average.
We use it to choose lambda (or the degree).

In [ ]:
def kfold_indices(n, k, seed=0):
    """List of k arrays of held-out indices that together cover 0..n-1 exactly once."""
    idx = np.random.default_rng(seed).permutation(n)
    return [np.sort(f) for f in np.array_split(idx, k)]


def cv_rmse(fit, predict, F, y, k=5, seed=0):
    """Average held-out RMSE. fit(F_train, y_train) -> model, predict(model, F_test) -> yhat."""
    F, y = np.asarray(F, float), np.asarray(y, float)
    errs = []
    for hold in kfold_indices(len(y), k, seed):
        keep = np.setdiff1d(np.arange(len(y)), hold)
        errs.append(rmse(y[hold], predict(fit(F[keep], y[keep]), F[hold])))
    return float(np.mean(errs))


def cv_choose_degree(z, y, max_degree=8, k=5):
    scores = []
    for d in range(1, max_degree + 1):
        scores.append(cv_rmse(lambda F, t: np.linalg.lstsq(np.column_stack([np.ones(len(t)), F]), t, rcond=None)[0],
                              lambda m, F: np.column_stack([np.ones(len(F)), F]) @ m,
                              poly_features(z, d), y, k))
    return int(np.argmin(scores)) + 1, np.array(scores)


def cv_choose_lambda(z, y, lams, degree=9, k=5):
    scores = [cv_rmse(lambda F, t, l=lam: ridge_fit(F, t, l), ridge_predict, poly_features(z, degree), y, k) for lam in lams]
    return float(lams[int(np.argmin(scores))]), np.array(scores)

## Step 4: bias and variance by refitting
Draw many training sets from the same farm, fit each, and look at the predictions at one fixed
rain value z0. The spread of those predictions is the **variance**; the gap between their mean
and the truth is the **bias**. Flexible models: small bias, large variance.

In [ ]:
def bias_variance(degree, z0=0.9, repeats=300, n_train=20, seed=0):
    """(bias, variance of the prediction at z0 over refits)."""
    preds = []
    for r in range(repeats):
        z, y = make_response(n_train, seed * 100000 + r)
        preds.append(float((poly_design([z0], degree) @ fit_poly(z, y, degree))[0]))
    preds = np.array(preds)
    return float(preds.mean() - truth(z0)), float(preds.var())

## Step 5: outliers and robust lines
One broken sensor reading can drag a least-squares line far off. Two robust alternatives we write
ourselves: the **Theil-Sen** line (slope = median of all pairwise slopes, intercept = median of
y - slope x) and a **Huber** fit (iteratively reweighted least squares: points with large
residuals get weight k s / |r|, others weight 1; here k = 1.345 and s is the median absolute
residual / 0.6745).

In [ ]:
def ols_line(x, y):
    X = np.column_stack([np.ones(len(x)), x])
    return tuple(float(v) for v in np.linalg.lstsq(X, np.asarray(y, float), rcond=None)[0])


def theil_sen_line(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    i, j = np.triu_indices(len(x), 1)
    keep = x[j] != x[i]
    slope = float(np.median((y[j] - y[i])[keep] / (x[j] - x[i])[keep]))
    return float(np.median(y - slope * x)), slope


def huber_line(x, y, k=1.345, iters=50):
    x, y = np.asarray(x, float), np.asarray(y, float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    for _ in range(iters):
        r = y - X @ beta
        s = max(np.median(np.abs(r - np.median(r))) / 0.6745, 1e-12)
        w = np.minimum(1.0, k * s / np.maximum(np.abs(r), 1e-12))
        sw = np.sqrt(w)
        beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
    return float(beta[0]), float(beta[1])


def with_outliers(seed=4, n=20):
    """Farm data (Session 2 style) with the three highest-rain seasons wrongly recorded as 0.5 t/ha."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(3.0, 9.0, n)
    y = 1.0 + 0.35 * x + rng.normal(0.0, 0.3, n)
    bad = np.argsort(x)[-3:]
    y = y.copy()
    y[bad] = 0.5
    return x, y

## Step 6: correlation is not causation
Hot days raise ice-cream sales and (through more swimming) drownings. Ice cream does not cause
drowning: temperature is a **confounder**. The *partial correlation* of A and B given T is the
correlation of what is left of A and of B after removing the straight-line effect of T. If it
vanishes, T explained the link.

In [ ]:
def make_confounded(n=500, seed=2):
    rng = np.random.default_rng(seed)
    temp = rng.normal(25.0, 5.0, n)                    # degrees C
    ice = 10.0 + 2.0 * temp + rng.normal(0.0, 5.0, n)  # sales index
    drown = 0.5 + 0.15 * temp + rng.normal(0.0, 0.8, n)
    return temp, ice, drown


def partial_corr(a, b, control):
    """Correlation of a and b after removing the linear effect of control from both."""
    def resid(v):
        c = ols_line(control, v)
        return np.asarray(v, float) - c[0] - c[1] * np.asarray(control, float)
    return float(np.corrcoef(resid(a), resid(b))[0, 1])

## Step 7: failure case, multicollinearity
If two inputs are almost copies of each other (say rain gauge A and rain gauge B), the model
cannot tell which deserves credit. The columns of X are nearly dependent, the condition number
of X^T X explodes, and the coefficients swing wildly from one data set to the next, but the
*predictions* (their sum) stay fine.

In [ ]:
def make_collinear(n=40, seed=0, gap=0.02):
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(3.0, 9.0, n)
    x2 = x1 + rng.normal(0.0, gap * 5, n)               # nearly the same gauge
    y = 1.0 + 0.35 * x1 + rng.normal(0.0, 0.3, n)       # truth uses rain once
    return np.column_stack([np.ones(n), x1, x2]), y


def collinear_experiment(repeats=500, gap=0.02):
    """Coefficient sd of x1, sd of the sum x1 + x2, sd of a prediction at (6, 6), condition number."""
    coefs, preds, conds = [], [], []
    for r in range(repeats):
        X, y = make_collinear(40, r, gap)
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        coefs.append(beta)
        preds.append(beta @ np.array([1.0, 6.0, 6.0]))
        conds.append(np.linalg.cond(X.T @ X))
    coefs = np.array(coefs)
    return {"sd_b1": float(coefs[:, 1].std()), "sd_sum": float((coefs[:, 1] + coefs[:, 2]).std()),
            "sd_pred": float(np.std(preds)), "cond": float(np.median(conds))}

## Step 8: recovery of the true coefficients and sensitivity
Because we generated the data, we can check the fit against the truth: a 95 percent range
beta +- 1.96 SE should contain the true slope in about 95 percent of repeated data sets. And SE of
the slope is sigma / sqrt(Sxx), so quadrupling n halves it, and doubling the noise doubles it.

In [ ]:
def slope_stats(n, sigma, repeats=2000, seed=0, a=1.0, b=0.35):
    """(fraction of repeats whose slope +- 1.96 SE holds b, mean SE of the slope, sd of the fitted slopes)."""
    rng = np.random.default_rng(seed)
    hit, ses, slopes = 0, [], []
    for _ in range(repeats):
        x = rng.uniform(3.0, 9.0, n)
        y = a + b * x + rng.normal(0.0, sigma, n)
        a_h, b_h = ols_line(x, y)
        s = math.sqrt(np.sum((y - a_h - b_h * x) ** 2) / (n - 2))
        se = s / math.sqrt(np.sum((x - x.mean()) ** 2))
        hit += abs(b_h - b) <= 1.96 * se
        ses.append(se)
        slopes.append(b_h)
    return hit / repeats, float(np.mean(ses)), float(np.std(slopes))

## Step 9: mini-project, build and audit a yield model
A synthetic farm data set of 60 seasons: rain, fertiliser, mean temperature. We build the model
by least squares, audit it (R^2, residual checks, cross-validation against a mean-only model and
against a too-flexible one), and report a 95 percent prediction interval for next season.

In [ ]:
FARM_COEF = np.array([0.5, 0.30, 0.10, -0.04])   # intercept, rain (100 mm), fertiliser (10 kg/ha), temperature (C)


def farm_dataset(n=60, seed=21, sigma=0.35):
    rng = np.random.default_rng(seed)
    rain = rng.uniform(3.0, 9.0, n)
    fert = rng.uniform(0.0, 10.0, n)
    temp = rng.normal(24.0, 2.5, n)
    X = np.column_stack([np.ones(n), rain, fert, temp])
    return X, X @ FARM_COEF + rng.normal(0.0, sigma, n)


def prediction_interval_multi(X, y, x0, level=0.95):
    """Prediction interval for one new season with inputs x0 (including the leading 1)."""
    n, p = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    s2 = np.sum((y - X @ beta) ** 2) / (n - p)
    lev = float(x0 @ np.linalg.solve(X.T @ X, x0))
    # t quantile by bisection on the normal-free approach: reuse a simple series-free Simpson integral
    c = math.exp(math.lgamma((n - p + 1) / 2) - math.lgamma((n - p) / 2)) / math.sqrt((n - p) * math.pi)

    def cdf(t):
        u = np.linspace(0.0, abs(t), 2001)
        d = c * (1 + u ** 2 / (n - p)) ** (-(n - p + 1) / 2)
        h = u[1] - u[0]
        return 0.5 + math.copysign(h / 3 * (d[0] + d[-1] + 4 * d[1:-1:2].sum() + 2 * d[2:-1:2].sum()), t)

    lo, hi = 0.0, 20.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if cdf(mid) < 0.5 + level / 2 else (lo, mid)
    half = (lo + hi) / 2 * math.sqrt(s2 * (1 + lev))
    yh = float(x0 @ beta)
    return yh, yh - half, yh + half


def _ls_fit(F, t):
    return np.linalg.lstsq(np.column_stack([np.ones(len(t)), F]), t, rcond=None)[0]


def _ls_predict(m, F):
    return np.column_stack([np.ones(len(F)), F]) @ m


def expand_quadratic(F):
    """Inputs plus all their squares and pairwise products (a much more flexible model)."""
    F = np.asarray(F, float)
    cols = [F[:, i] for i in range(F.shape[1])]
    cols += [F[:, i] * F[:, j] for i in range(F.shape[1]) for j in range(i, F.shape[1])]
    return np.column_stack(cols)


def farm_audit(X, y, k=6):
    """Cross-validated RMSE (t/ha) of: mean only, rain only, three inputs, three inputs with squares and products."""
    F = X[:, 1:]
    zero = lambda F_, t: float(np.mean(t))
    return {
        "mean only": cv_rmse(zero, lambda m, F_: np.full(len(F_), m), F, y, k),
        "rain only": cv_rmse(_ls_fit, _ls_predict, F[:, :1], y, k),
        "three inputs": cv_rmse(_ls_fit, _ls_predict, F, y, k),
        "too flexible": cv_rmse(_ls_fit, _ls_predict, expand_quadratic(F), y, k),
    }


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    tr, te = degree_curve()
    print("train RMSE by degree:", tr.round(2))
    print("test  RMSE by degree:", te.round(2))
    print("best degree on test:", int(np.argmin(te)), "with RMSE", round(float(te.min()), 3), "(noise sd 0.30)")
    lams = np.array([0, 0.01, 0.1, 1, 10, 100, 1000])
    big, tte = ridge_path(lams)
    print("ridge: max |coef|", big.round(2), "test RMSE", tte.round(3))
    zt, yt = make_response(20, 1)
    print("CV degree:", cv_choose_degree(zt, yt)[0], " CV lambda:", cv_choose_lambda(zt, yt, np.array([1e-3, 1e-2, 1e-1, 1, 10, 100]))[0])
    for d in (1, 3, 9):
        print("bias, variance at z0=0.9, degree", d, np.round(bias_variance(d), 4))
    x, y = with_outliers()
    print("ols", np.round(ols_line(x, y), 3), "theil-sen", np.round(theil_sen_line(x, y), 3), "huber", np.round(huber_line(x, y), 3), "(true 1.0, 0.35)")
    t, i, d = make_confounded()
    print("raw corr", round(float(np.corrcoef(i, d)[0, 1]), 3), "partial given temp", round(partial_corr(i, d, t), 3))
    print("collinear:", {k: round(v, 3) for k, v in collinear_experiment().items()})
    for n, s in ((25, 0.3), (100, 0.3), (25, 0.6)):
        print("slope stats n, sigma", n, s, np.round(slope_stats(n, s), 4))
    X, yy = farm_dataset()
    beta = np.linalg.lstsq(X, yy, rcond=None)[0]
    print("farm coefficients", beta.round(3), "R2", round(1 - np.sum((yy - X @ beta) ** 2) / np.sum((yy - yy.mean()) ** 2), 3))
    print("audit:", {k: round(v, 3) for k, v in farm_audit(X, yy).items()})
    print("next season (550 mm, 60 kg/ha, 25 C):", np.round(prediction_interval_multi(X, yy, np.array([1.0, 5.5, 6.0, 25.0])), 2))

    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    axs[0].plot(range(len(tr)), tr, "o-", label="training error")
    axs[0].plot(range(len(te)), te, "s-", label="test error")
    axs[0].set_ylim(0, 1.5)
    axs[0].axvline(int(np.argmin(te)), color="gray", ls=":")
    axs[0].set_xlabel("polynomial degree")
    axs[0].set_ylabel("RMSE (t/ha)")
    axs[0].set_title("Test error is U-shaped")
    axs[0].legend()
    axs[1].semilogx(lams[1:], tte[1:], "o-")
    axs[1].axhline(tte[0], color="tab:red", ls="--", label="lambda = 0 (least squares)")
    axs[1].set_xlabel("ridge lambda")
    axs[1].set_ylabel("test RMSE (t/ha)")
    axs[1].set_title("Ridge tames degree 9")
    axs[1].legend()
    xo, yo = with_outliers()
    g = np.linspace(3, 9, 10)
    axs[2].scatter(xo, yo)
    for name, f in (("least squares", ols_line), ("Theil-Sen", theil_sen_line), ("Huber", huber_line)):
        a_, b_ = f(xo, yo)
        axs[2].plot(g, a_ + b_ * g, label=name)
    axs[2].set_xlabel("rainfall (hundreds of mm)")
    axs[2].set_ylabel("yield (t/ha)")
    axs[2].set_title("Three wrong records drag least squares")
    axs[2].legend()
    for ax in axs:
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "overfit_ridge_robust.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Run `degree_curve(seed=5)` and report the best degree. Does the U shape survive a different seed?
2. **Stretch.** Write `loo_cv` (leave-one-out, k = n) with `kfold_indices` and compare its choice of lambda with the 5-fold choice.
3. **Explain.** Explain why standardising the features matters for ridge but not for ordinary least squares.

## Assessment
- **A. Conceptual.** A colleague picks the polynomial degree with the smallest training error and reports it. Why is that wrong, and what should be done instead?
- **B. Numerical.** For the three points x = -1, 0, 1 with y = 0, 1, 5, find the ridge slope in original units for lambda = 6.
- **C. Graph and simulation interpretation.** In the app, at degree 9, compare the expected test error at log10 lambda = -4 and at -1. What does each tell you about the fit?
- **D. Design and optimisation challenge.** Design: for the farm data set of the mini-project, you can deploy the mean only, rain only, the three inputs, or the too-flexible model. Choose by cross-validation and give the forecast for 550 mm, 60 kg/ha, 25 degrees C with its interval. What do you tell the farmer about it?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    tr, te = L.degree_curve()
    checks.append(("training error never rises with degree", bool(np.all(np.diff(tr) <= 1e-9))))
    best = int(np.argmin(te))
    checks.append((f"test error is U-shaped: best degree is 3 (got {best}), RMSE 0.365", best == 3 and abs(te[best] - 0.365) < 5e-3))
    checks.append(("degree 12 test error (5.1 t/ha) is more than 10 times the best; training error is 0.22",
                   te[12] > 10 * te[best] and abs(tr[12] - 0.22) < 0.01))
    checks.append(("degree 0 and 12 are both worse than degree 3", te[0] > te[best] and te[12] > te[best]))

    # worked example: ridge by hand on three points
    F = np.array([[-1.0], [0.0], [1.0]])
    y = np.array([0.0, 1.0, 5.0])
    sd = np.sqrt(2.0 / 3.0)
    m0, m3 = L.ridge_fit(F, y, 0.0), L.ridge_fit(F, y, 3.0)
    checks.append(("worked example: ridge by hand, lambda 0 gives standardised slope 6.1237/3 = 2.0412, raw slope 2.5",
                   abs(m0["beta"][0] - 6.1237 / 3) < 1e-3 and abs(m0["beta"][0] / sd - 2.5) < 1e-9))
    checks.append(("worked example: lambda = 3 halves the slope (factor 3/(3+3)): raw slope 1.25",
                   abs(m3["beta"][0] / sd - 1.25) < 1e-9))

    zt, yt = L.make_response(20, 1)
    zs, ys = L.make_response(500, 1001)
    mr = L.ridge_fit(L.poly_features(zt, 9), yt, 0.0)
    ls = L.poly_design(zs, 9) @ L.fit_poly(zt, yt, 9)
    checks.append(("lambda = 0 recovers least squares (same predictions as lstsq)", np.allclose(L.ridge_predict(mr, L.poly_features(zs, 9)), ls, atol=1e-5)))
    Fs, _, _ = L.standardise(L.poly_features(zt, 9))
    mm = L.ridge_fit(L.poly_features(zt, 9), yt, 2.0)
    aug = np.linalg.lstsq(np.vstack([Fs, np.sqrt(2.0) * np.eye(9)]), np.r_[yt - yt.mean(), np.zeros(9)], rcond=None)[0]
    checks.append(("closed form agrees with least squares on data augmented by sqrt(lambda) I", np.allclose(mm["beta"], aug, atol=1e-8)))
    lams = np.array([0, 0.01, 0.1, 1, 10, 100, 1000])
    big, tte = L.ridge_path(lams)
    checks.append(("ridge shrinks coefficients: largest |coef| falls from 26.0 to 0.02 as lambda grows", bool(np.all(np.diff(big) < 0)) and abs(big[0] - 26.0) < 0.1 and abs(big[-1] - 0.02) < 0.01))
    checks.append(("ridge test RMSE: 0.596 at lambda 0, best 0.371 at lambda 0.1, 1.16 at lambda 1000",
                   int(np.argmin(tte)) == 2 and abs(tte[0] - 0.596) < 5e-3 and abs(tte[2] - 0.371) < 5e-3 and abs(tte[-1] - 1.16) < 0.02))

    folds = L.kfold_indices(23, 5, seed=3)
    allidx = np.sort(np.concatenate(folds))
    checks.append(("edge: k-fold folds cover every index exactly once (n = 23, k = 5)", np.array_equal(allidx, np.arange(23)) and max(map(len, folds)) - min(map(len, folds)) <= 1))
    lam, _ = L.cv_choose_lambda(zt, yt, np.array([1e-3, 1e-2, 1e-1, 1, 10, 100]))
    deg, _ = L.cv_choose_degree(zt, yt)
    checks.append((f"cross-validation picks lambda 0.1 (got {lam}) and degree 6 (got {deg}) on these 20 seasons", lam == 0.1 and deg == 6))

    bv = {d: L.bias_variance(d) for d in (1, 3, 9)}
    checks.append(("variance at z0 = 0.9 grows with flexibility: degrees 1, 3, 9 give 0.057, 0.066, about 28000",
                   bv[1][1] < bv[3][1] < bv[9][1] and abs(bv[1][1] - 0.057) < 5e-3 and abs(bv[3][1] - 0.066) < 5e-3 and bv[9][1] > 1e4))
    checks.append(("bias: degree 1 is biased (0.54), degree 3 is nearly unbiased (-0.003)", abs(bv[1][0] - 0.537) < 0.01 and abs(bv[3][0]) < 0.02))

    x, yo = L.with_outliers()
    truth = np.array([1.0, 0.35])
    eo, et, eh = (np.abs(np.array(f(x, yo)) - truth)[1] for f in (L.ols_line, L.theil_sen_line, L.huber_line))
    checks.append(("outliers: slope 0.009 by least squares, 0.215 Theil-Sen, 0.290 Huber (true 0.35)",
                   abs(L.ols_line(x, yo)[1] - 0.009) < 2e-3 and abs(L.theil_sen_line(x, yo)[1] - 0.215) < 2e-3 and abs(L.huber_line(x, yo)[1] - 0.290) < 2e-3 and et < eo and eh < eo))
    xc = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    checks.append(("robust lines reproduce a perfect line y = 2 + 3x exactly",
                   np.allclose(L.theil_sen_line(xc, 2 + 3 * xc), [2, 3]) and np.allclose(L.huber_line(xc, 2 + 3 * xc), [2, 3], atol=1e-8)))

    t, ice, dr = L.make_confounded()
    raw = np.corrcoef(ice, dr)[0, 1]
    part = L.partial_corr(ice, dr, t)
    checks.append((f"confounder: ice cream and drowning correlate {raw:.2f} but partial correlation given temperature is {part:.2f}", abs(raw - 0.596) < 5e-3 and abs(part) < 0.1))

    ce = L.collinear_experiment()
    checks.append(("multicollinearity: coefficient sd 0.50 but sd of the sum 0.028 and of the prediction 0.049; condition number about 17000",
                   abs(ce["sd_b1"] - 0.502) < 0.01 and ce["sd_pred"] < 0.06 and ce["sd_b1"] > 8 * ce["sd_pred"] and 1e4 < ce["cond"] < 3e4))

    c25, se25, sd25 = L.slope_stats(25, 0.3)
    c100, se100, _ = L.slope_stats(100, 0.3)
    _, se_noisy, _ = L.slope_stats(25, 0.6)
    checks.append((f"recovery: slope +- 1.96 SE holds the true 0.35 in {c25:.3f} of 2000 repeats (about 95 percent)", 0.935 <= c25 <= 0.965 and abs(se25 - sd25) / sd25 < 0.05))
    checks.append(("sample size: n 25 -> 100 halves the SE (0.0356 -> 0.0174); doubling the noise doubles it (0.0713)",
                   abs(se25 - 0.0356) < 5e-4 and abs(se100 - 0.0174) < 5e-4 and abs(se_noisy / se25 - 2.0) < 0.05))

    X, yy = L.farm_dataset()
    beta = np.linalg.lstsq(X, yy, rcond=None)[0]
    checks.append(("mini-project: coefficients 0.965, 0.300, 0.093, -0.061", np.allclose(beta, [0.965, 0.300, 0.093, -0.061], atol=2e-3)))
    aud = L.farm_audit(X, yy)
    checks.append(("audit (6-fold RMSE t/ha): mean 0.724, rain only 0.440, three inputs 0.343, too flexible 0.381",
                   abs(aud["mean only"] - 0.724) < 2e-3 and abs(aud["rain only"] - 0.440) < 2e-3 and abs(aud["three inputs"] - 0.343) < 2e-3 and abs(aud["too flexible"] - 0.381) < 2e-3))
    yh, lo, hi = L.prediction_interval_multi(X, yy, np.array([1.0, 5.5, 6.0, 25.0]))
    checks.append(("next season (550 mm, 60 kg/ha, 25 C): 1.64 t/ha, 95 percent interval 0.95 to 2.32", abs(yh - 1.64) < 0.01 and abs(lo - 0.95) < 0.01 and abs(hi - 2.32) < 0.01))
    checks.append(("training RMSE is about 0.32 at degree 3 and test RMSE at degree 12 is 5.1", abs(tr[3] - 0.32) < 0.01 and abs(te[12] - 5.1) < 0.05))
    sd = np.sqrt(2.0 / 3.0)
    m6 = L.ridge_fit(F, y, 6.0)
    checks.append(("question B: lambda = 6 gives raw slope 2.5 / 3 = 0.833", abs(m6["beta"][0] / sd - 0.8333) < 1e-3))
    zr, yr = np.round(zt, 4), np.round(yt, 4)

    def app_numbers(d, ll):
        m = L.ridge_fit(L.poly_features(zr, d), yr, 10.0 ** ll)
        f = lambda t: L.ridge_predict(m, L.poly_features(t, d))
        return L.rmse(yr, f(zr)), L.expected_test_rmse(f)
    o, t9 = app_numbers(9, -4), app_numbers(9, -1)
    checks.append(("app: degree 9 gives train 0.250 and expected test 0.832 at log10 lambda -4, train 0.291 and test 0.357 at -1",
                   abs(o[0] - 0.250) < 2e-3 and abs(o[1] - 0.832) < 3e-3 and abs(t9[0] - 0.291) < 2e-3 and abs(t9[1] - 0.357) < 2e-3))
    checks.append(("app challenge range: degree 9 with log10 lambda -1 and -0.5 keeps expected test below 0.4",
                   app_numbers(9, -1)[1] < 0.4 and app_numbers(9, -0.5)[1] < 0.4))
    checks.append(("ridge never sets a coefficient exactly to zero (milestone baseline)", np.count_nonzero(L.ridge_fit(L.poly_features(zt, 9), yt, 0.1)["beta"] == 0) == 0))
    checks.append(("mini-project R squared is 0.797", abs(1 - np.sum((yy - X @ beta) ** 2) / np.sum((yy - yy.mean()) ** 2) - 0.797) < 5e-4))
    rng = np.random.default_rng(3)
    xs = rng.uniform(3, 9, 200)
    ys = 1 + 0.35 * xs + rng.normal(0, 0.3, 200)
    a_, b_ = L.ols_line(xs, ys)
    xn = rng.uniform(3, 9, 20000)
    yn = 1 + 0.35 * xn + rng.normal(0, 0.3, 20000)
    frac = float(np.mean(yn < a_ + b_ * xn))
    checks.append((f"milestone baseline: the least-squares line has {frac:.2f} of fresh seasons below it", abs(frac - 0.51) < 0.01))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
